# Day 3, Session 3 — Capstone: Multi-Agent Consulting System

You are the McKinsey engineer building the demo. A Partner wants a working prototype: an **Engagement Manager agent** takes a complex consulting question, structures a plan, and hands it off to a **specialist agent** who answers using the knowledge base you built in Session 1.

Two milestones. One working system. Two-minute demo at the end.

In [1]:
!pip install -q openai langchain langchain-openai langchain-core langgraph chromadb python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


## Your role

You are the engineer. The Partner and the engagement team are the users — they will type questions and read the output your system produces. Your engineering decisions: how the EM structures its plan, what persona the specialist uses, and how the RAG tool retrieves and formats context.

The corpus is the same consulting knowledge base you indexed in D3S1 (frameworks, playbooks, industry reports). The specialist agent calls it as a tool to ground its answers in real content.

In [2]:
import os
import json
import numpy as np
from dotenv import load_dotenv
load_dotenv()

import openai
import chromadb
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

client = openai.OpenAI()
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=0)

# ============================================================
# Consulting Knowledge Base (same corpus as D3S1)
# ============================================================
knowledge_docs = [
    {
        "text": "McKinsey's post-merger integration (PMI) framework emphasizes three pillars: capturing synergies within the first 100 days, aligning organizational culture through structured leadership workshops, and establishing a clean-room integration management office (IMO). Research across 200+ acquisitions shows that companies capturing 80% of synergies in year one outperform peers by 15% in total shareholder returns.",
        "source": "pmi_framework_2024.md",
        "metadata": {"practice_area": "M&A", "industry": "Cross-industry"}
    },
    {
        "text": "Digital transformation in manufacturing requires a phased approach: Phase 1 -- Assess digital maturity across the value chain using McKinsey's Digital Quotient (DQ) diagnostic. Phase 2 -- Prioritize use cases by impact and feasibility (typically predictive maintenance, demand sensing, and quality analytics yield highest ROI). Phase 3 -- Build a scalable data platform and upskill the workforce.",
        "source": "digital_manufacturing_playbook.md",
        "metadata": {"practice_area": "Digital", "industry": "Manufacturing"}
    },
    {
        "text": "Operations excellence programs typically deliver 15-25% cost reduction in the first 18 months. McKinsey's approach begins with a diagnostic spanning procurement, production, and logistics. Lean management principles are combined with advanced analytics to identify waste. A critical success factor is embedding a performance management infrastructure -- daily management systems, KPI cascades, and capability-building academies.",
        "source": "ops_excellence_guide.md",
        "metadata": {"practice_area": "Operations", "industry": "Cross-industry"}
    },
    {
        "text": "CEO briefing templates should follow the Situation-Complication-Resolution (SCR) structure. Open with a one-sentence situation framing the strategic context. Introduce the complication -- the specific challenge or opportunity requiring action. Close with the resolution -- McKinsey's recommended path forward with 2-3 actionable next steps. Keep the core narrative to 5-7 pages maximum.",
        "source": "ceo_briefing_template.md",
        "metadata": {"practice_area": "Leadership", "industry": "Cross-industry"}
    },
    {
        "text": "Supply chain resilience has become a board-level priority following recent global disruptions. McKinsey's supply chain diagnostic evaluates five dimensions: supplier diversification, inventory buffering strategy, logistics network flexibility, digital visibility (control tower maturity), and workforce agility. Best-in-class companies maintain dual sourcing for critical components and hold 4-6 weeks of safety stock for high-risk SKUs.",
        "source": "supply_chain_resilience_report.md",
        "metadata": {"practice_area": "Operations", "industry": "Manufacturing"}
    }
]

# Index in ChromaDB
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("capstone_knowledge")
texts = [d["text"] for d in knowledge_docs]
emb_response = client.embeddings.create(model="text-embedding-3-small", input=texts)
embeddings = [item.embedding for item in emb_response.data]
collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=[f"doc_{i}" for i in range(len(texts))],
    metadatas=[{**d["metadata"], "source": d["source"]} for d in knowledge_docs]
)
print(f"Indexed {len(texts)} documents in ChromaDB. Setup complete!")

Indexed 5 documents in ChromaDB. Setup complete!


---
## Milestone 1: Engagement Manager Agent

Build an Engagement Manager agent that takes a complex consulting question, identifies the key workstream, and returns a structured JSON plan naming the workstream and the specialist to assign.

The EM does not answer the question. It plans. The output is a JSON object that Milestone 2 consumes.

**Requirements:**
- Accept a consulting question (e.g., PE due diligence, market entry assessment, digital transformation strategy)
- Return JSON with: `workstream` (what to investigate), `specialist` (who should do it), `hypothesis` (what the specialist should test)
- Use `response_format={"type": "json_object"}` for reliable structured output

In [3]:
# Milestone 1 — Engagement Manager Agent
#
# TODO: Build an EM agent that takes a consulting question and returns
# a structured JSON plan with: workstream, specialist, hypothesis.
#
# Hint: System prompt should instruct the LLM to act as a McKinsey EM
# Hint: Tell the LLM to return JSON with exactly these keys: workstream, specialist, hypothesis
# Hint: Available specialists: strategy_analyst, financial_analyst, operations_expert, industry_researcher
# Hint: Use temperature=0 for deterministic planning

def engagement_manager(question: str) -> dict:
    """Takes a consulting question, returns a structured engagement plan as JSON."""
    # YOUR CODE HERE
    pass


# --- Test ---
# plan = engagement_manager(
#     "Our PE client is evaluating a $500M acquisition of a healthcare IT platform. "
#     "Assess the target's competitive positioning and growth trajectory."
# )
# print(json.dumps(plan, indent=2))
# assert "workstream" in plan and "specialist" in plan and "hypothesis" in plan

**Expected output shape** (your exact wording will vary):
```json
{
  "workstream": "Competitive positioning and growth trajectory analysis of the healthcare IT target",
  "specialist": "strategy_analyst",
  "hypothesis": "The target holds a defensible market position with above-market growth driven by ..."
}
```

---
## Milestone 2: Specialist Agent with RAG Tool

Build a specialist agent that receives the EM's assignment, calls a RAG tool over the consulting knowledge base, and returns an answer with citations.

The RAG tool is a function decorated with `@tool` that embeds the query, searches the ChromaDB collection from the setup cell, and returns the top chunks. The specialist uses those chunks as context to produce a grounded, cited answer.

**Requirements:**
- Define a RAG retrieval tool using the `@tool` decorator
- The tool searches the ChromaDB collection and returns relevant chunks with source metadata
- The specialist agent receives the EM's workstream and hypothesis, calls the RAG tool, and writes an answer
- The answer includes source citations (e.g., [Source: pmi_framework_2024.md])

In [4]:
# Milestone 2 — Specialist Agent with RAG Tool
#
# Step 1: Define a @tool function that searches ChromaDB.
#   - Embed the query using client.embeddings.create()
#   - Search collection.query(query_embeddings=[...], n_results=3)
#   - Format results as "[Source: filename] chunk_text" for each hit
#   - Return the formatted string
#
# Step 2: Build a specialist agent function.
#   - Takes workstream and hypothesis from the EM's plan
#   - Calls the RAG tool to get context
#   - Sends context + assignment to the LLM with a specialist system prompt
#   - Returns the LLM's cited answer
#
# Hint: The specialist system prompt should say "cite sources using [Source: ...]" 
# Hint: Include the workstream and hypothesis in the user message

@tool
def search_knowledge_base(query: str) -> str:
    """Search the McKinsey consulting knowledge base for relevant frameworks, playbooks, and reports."""
    # YOUR CODE HERE
    pass


def specialist_agent(workstream: str, hypothesis: str) -> str:
    """A consulting specialist that uses RAG to answer workstream questions with citations."""
    # YOUR CODE HERE
    pass


# --- End-to-End Test ---
# plan = engagement_manager(
#     "Our PE client is evaluating a $500M acquisition of a healthcare IT platform. "
#     "Assess the target's competitive positioning and growth trajectory."
# )
# print("=== EM Plan ===")
# print(json.dumps(plan, indent=2))
# print()
# answer = specialist_agent(plan["workstream"], plan["hypothesis"])
# print("=== Specialist Answer ===")
# print(answer)

**Expected output shape:**

The specialist's answer should:
- Reference content from the knowledge base (PMI framework, operations excellence, etc.)
- Include inline citations like `[Source: pmi_framework_2024.md]`
- Be structured and actionable — the Partner will read this

---
## Presentation — 2 minutes

Demo your working system. Run a consulting question end-to-end: EM plans, specialist retrieves and answers.

Name **one design decision** you made. Examples: “I used temperature 0 for the EM because the plan needs to be deterministic.” “I set k=3 for retrieval because more chunks diluted the answer quality.”

No slides. Just the running notebook and one sentence about a tradeoff.